# Experiment 7: categorical feature standardisation audit

This is a read-only audit. It identifies case, whitespace, hyphen, underscore, and punctuation variants in categorical/location columns, measures whether they affect train/test category coverage and HDB location matches, and exports the evidence for review. It does **not** change `preprocessing.py`.

In [1]:
from pathlib import Path
import os
import sys
import re
import numpy as np
import pandas as pd

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from preprocessing import load_datasets, normalize_key

_, datasets = load_datasets(project_root / 'data')
train_raw = datasets['train'].copy()
test_raw = datasets['test'].copy()
hdb_raw = datasets['hdb'].copy()
print(f'Train: {train_raw.shape}; test: {test_raw.shape}; HDB lookup: {hdb_raw.shape}')


Train: (150000, 11); test: (50000, 10); HDB lookup: (9464, 11)


In [2]:
# Each normaliser is deliberately more aggressive than the preceding one.
# The audit compares their effects; it does not apply any of them to model data.
def as_text(values):
    return values.fillna('<MISSING>').astype(str)

def case_whitespace(values):
    return as_text(values).str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def separator_fold(values):
    return case_whitespace(values).str.replace(r'[-_/]+', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

def alphanumeric_fold(values):
    return case_whitespace(values).str.replace(r'[^a-z0-9]+', '', regex=True)

NORMALISERS = {
    'raw': as_text,
    'case_whitespace': case_whitespace,
    'separator_fold': separator_fold,
    'alphanumeric_fold': alphanumeric_fold,
}


In [3]:
MAIN_CATEGORICAL_COLUMNS = ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FURNISHED']
main_columns = [column for column in MAIN_CATEGORICAL_COLUMNS if column in train_raw.columns]
combined_main = pd.concat([
    train_raw.assign(_source='train'),
    test_raw.assign(_source='test'),
], ignore_index=True)

summary_rows = []
collision_rows = []
for column in main_columns:
    values = combined_main[column]
    for normaliser_name, normaliser in NORMALISERS.items():
        normalised = normaliser(values)
        raw_variants_per_key = pd.DataFrame({'raw': as_text(values), 'key': normalised}).groupby('key')['raw'].nunique()
        summary_rows.append({
            'dataset': 'main_train_and_test', 'column': column, 'normaliser': normaliser_name,
            'unique_values': normalised.nunique(dropna=False),
            'keys_with_multiple_raw_variants': int((raw_variants_per_key > 1).sum()),
            'rows_in_colliding_keys': int(normalised.isin(raw_variants_per_key[raw_variants_per_key > 1].index).sum()),
        })
        if normaliser_name != 'raw':
            variants = pd.DataFrame({'raw': as_text(values), 'key': normalised}).drop_duplicates()
            for key, group in variants.groupby('key'):
                raw_values = sorted(group['raw'].tolist())
                if len(raw_values) > 1:
                    collision_rows.append({
                        'dataset': 'main_train_and_test', 'column': column, 'normaliser': normaliser_name,
                        'canonical_key': key, 'raw_variants': ' | '.join(raw_values[:12]),
                        'variant_count': len(raw_values),
                    })

main_summary = pd.DataFrame(summary_rows)
main_collisions = pd.DataFrame(collision_rows).sort_values(['column', 'normaliser', 'variant_count'], ascending=[True, True, False])
display(main_summary.pivot(index='column', columns='normaliser', values='unique_values'))
display(main_collisions.head(100))


normaliser,alphanumeric_fold,case_whitespace,raw,separator_fold
column,,,,
BLOCK,2731,2731,2731,2731
FLAT_MODEL,20,20,20,20
FLAT_TYPE,6,11,11,6
FURNISHED,1,1,1,1
STREET,576,576,1150,576
TOWN,26,26,26,26


,dataset,column,normaliser,canonical_key,raw_variants,variant_count
1727,main_train_and_test,FLAT_TYPE,alphanumeric_fold,1room,1 room | 1-room,2
1728,main_train_and_test,FLAT_TYPE,alphanumeric_fold,2room,2 room | 2-room,2
1729,main_train_and_test,FLAT_TYPE,alphanumeric_fold,3room,3 room | 3-room,2
1730,main_train_and_test,FLAT_TYPE,alphanumeric_fold,4room,4 room | 4-room,2
1731,main_train_and_test,FLAT_TYPE,alphanumeric_fold,5room,5 room | 5-room,2
...,...,...,...,...,...,...
1233,main_train_and_test,STREET,alphanumeric_fold,bukitbatokstreet41,Bukit Batok Street 41 | bukit batok street 41,2
1234,main_train_and_test,STREET,alphanumeric_fold,bukitbatokstreet51,Bukit Batok Street 51 | bukit batok street 51,2
1235,main_train_and_test,STREET,alphanumeric_fold,bukitbatokstreet52,Bukit Batok Street 52 | bukit batok street 52,2
1236,main_train_and_test,STREET,alphanumeric_fold,bukitbatokwestavenue2,Bukit Batok West Avenue 2 | bukit batok west a...,2


In [4]:
# Check whether normalisation changes how many test categories are unseen during training.
coverage_rows = []
for column in main_columns:
    for normaliser_name, normaliser in NORMALISERS.items():
        train_values = set(normaliser(train_raw[column]))
        test_values = normaliser(test_raw[column])
        unseen_mask = ~test_values.isin(train_values)
        coverage_rows.append({
            'column': column, 'normaliser': normaliser_name,
            'train_unique': len(train_values), 'test_unique': test_values.nunique(),
            'unseen_test_categories': test_values[unseen_mask].nunique(),
            'test_rows_with_unseen_category': int(unseen_mask.sum()),
            'test_row_share_unseen': float(unseen_mask.mean()),
        })
coverage_report = pd.DataFrame(coverage_rows)
display(coverage_report.sort_values(['column', 'normaliser']))

# Show exact raw train/test variants for the columns where separator folding improves coverage.
for column in main_columns:
    raw_unseen = coverage_report[(coverage_report['column'] == column) & (coverage_report['normaliser'] == 'raw')].iloc[0]['test_rows_with_unseen_category']
    folded_unseen = coverage_report[(coverage_report['column'] == column) & (coverage_report['normaliser'] == 'separator_fold')].iloc[0]['test_rows_with_unseen_category']
    if folded_unseen < raw_unseen:
        print(f'\n{column}: separator folding removes {int(raw_unseen - folded_unseen):,} unseen test rows')
        print(sorted(set(as_text(test_raw[column])) - set(as_text(train_raw[column])))[:30])


,column,normaliser,train_unique,test_unique,unseen_test_categories,test_rows_with_unseen_category,test_row_share_unseen
7,BLOCK,alphanumeric_fold,2720,2660,11,36,0.00072
5,BLOCK,case_whitespace,2720,2660,11,36,0.00072
4,BLOCK,raw,2720,2660,11,36,0.00072
6,BLOCK,separator_fold,2720,2660,11,36,0.00072
19,FLAT_MODEL,alphanumeric_fold,20,19,0,0,0.00000
17,FLAT_MODEL,case_whitespace,20,19,0,0,0.00000
16,FLAT_MODEL,raw,20,19,0,0,0.00000
18,FLAT_MODEL,separator_fold,20,19,0,0,0.00000
15,FLAT_TYPE,alphanumeric_fold,6,6,0,0,0.00000
13,FLAT_TYPE,case_whitespace,11,6,0,0,0.00000



STREET: separator folding removes 6 unseen test rows
['Ang Mo Kio Street 23', 'Bukit Batok Street 41', 'Jurong West Street 51', 'Yishun Street 42']


In [5]:
# Audit every HDB text column. POSTAL_CODE is an identifier, so inspect it for formatting only;
# it should not be one-hot encoded as an ordinary category without a separate modelling decision.
HDB_CATEGORICAL_COLUMNS = [
    column for column in hdb_raw.columns
    if pd.api.types.is_string_dtype(hdb_raw[column]) or hdb_raw[column].dtype == object
]
hdb_summary_rows = []
hdb_collision_rows = []
for column in HDB_CATEGORICAL_COLUMNS:
    for normaliser_name, normaliser in NORMALISERS.items():
        normalised = normaliser(hdb_raw[column])
        variants = pd.DataFrame({'raw': as_text(hdb_raw[column]), 'key': normalised}).drop_duplicates()
        raw_variants_per_key = variants.groupby('key')['raw'].nunique()
        hdb_summary_rows.append({
            'dataset': 'hdb_lookup', 'column': column, 'normaliser': normaliser_name,
            'unique_values': normalised.nunique(),
            'keys_with_multiple_raw_variants': int((raw_variants_per_key > 1).sum()),
        })
        if normaliser_name != 'raw':
            for key, group in variants.groupby('key'):
                raw_values = sorted(group['raw'].tolist())
                if len(raw_values) > 1:
                    hdb_collision_rows.append({
                        'column': column, 'normaliser': normaliser_name, 'canonical_key': key,
                        'raw_variants': ' | '.join(raw_values[:12]), 'variant_count': len(raw_values),
                    })
hdb_summary = pd.DataFrame(hdb_summary_rows)
hdb_collisions = pd.DataFrame(hdb_collision_rows, columns=['column', 'normaliser', 'canonical_key', 'raw_variants', 'variant_count'])
display(hdb_summary.pivot(index='column', columns='normaliser', values='unique_values'))
display(hdb_collisions.head(100))

def location_coverage(normaliser_name, normaliser):
    listings = pd.concat([train_raw[['BLOCK', 'STREET']], test_raw[['BLOCK', 'STREET']]], ignore_index=True).copy()
    lookup = hdb_raw[['BLOCK', 'STREET']].copy()
    listings['block_key'] = normaliser(listings['BLOCK'])
    listings['street_key'] = normaliser(listings['STREET'])
    lookup['block_key'] = normaliser(lookup['BLOCK'])
    lookup['street_key'] = normaliser(lookup['STREET'])
    lookup_keys = pd.MultiIndex.from_frame(lookup[['block_key', 'street_key']].drop_duplicates())
    listing_keys = pd.MultiIndex.from_frame(listings[['block_key', 'street_key']])
    matched = listing_keys.isin(lookup_keys)
    return {
        'normaliser': normaliser_name, 'listing_rows': len(listings),
        'matched_rows': int(matched.sum()), 'match_rate': float(matched.mean()),
        'unmatched_unique_locations': int(pd.MultiIndex.from_frame(listings.loc[~matched, ['block_key', 'street_key']]).nunique()),
    }

location_report = pd.DataFrame([location_coverage(name, function) for name, function in NORMALISERS.items()])
display(location_report)
print('`case_whitespace` is equivalent to the current preprocessing.normalize_key behaviour for these fields.')


normaliser,alphanumeric_fold,case_whitespace,raw,separator_fold
column,,,,
BLOCK,2735,2735,2735,2735
PLANNING_AREA,31,31,31,31
POSTAL_CODE,9180,9180,9180,9180
REGION,5,5,5,5
STREET,576,576,576,576
SUBZONE,156,156,156,156
TOWN,26,26,26,26


,column,normaliser,canonical_key,raw_variants,variant_count


,normaliser,listing_rows,matched_rows,match_rate,unmatched_unique_locations
0,raw,200000,158737,0.793685,8565
1,case_whitespace,200000,200000,1.000000,0
2,separator_fold,200000,200000,1.000000,0
3,alphanumeric_fold,200000,200000,1.000000,0


`case_whitespace` is equivalent to the current preprocessing.normalize_key behaviour for these fields.


In [6]:
# FLOOR_AREA_SQM is numeric, not categorical. This only checks whether it needs cleaning/coercion.
area_rows = []
for source_name, frame in [('train', train_raw), ('test', test_raw)]:
    original = frame['FLOOR_AREA_SQM']
    numeric = pd.to_numeric(original, errors='coerce')
    area_rows.append({
        'source': source_name, 'rows': len(frame), 'null_or_non_numeric': int(numeric.isna().sum()),
        'min_sqm': numeric.min(), 'max_sqm': numeric.max(),
        'non_positive_values': int((numeric <= 0).sum()),
    })
area_report = pd.DataFrame(area_rows)
display(area_report)


,source,rows,null_or_non_numeric,min_sqm,max_sqm,non_positive_values
0,train,150000,0,31.0,215.0,0
1,test,50000,0,31.0,215.0,0


In [7]:
# Persist evidence only; no preprocessing or source-data file is changed by this notebook.
main_summary.to_csv(experiment_dir / 'exp_7_categorical_normalisation_summary.csv', index=False)
main_collisions.to_csv(experiment_dir / 'exp_7_categorical_collision_examples.csv', index=False)
coverage_report.to_csv(experiment_dir / 'exp_7_categorical_train_test_coverage.csv', index=False)
hdb_summary.to_csv(experiment_dir / 'exp_7_hdb_categorical_summary.csv', index=False)
hdb_collisions.to_csv(experiment_dir / 'exp_7_hdb_categorical_collision_examples.csv', index=False)
location_report.to_csv(experiment_dir / 'exp_7_location_key_coverage.csv', index=False)
area_report.to_csv(experiment_dir / 'exp_7_floor_area_sanity.csv', index=False)

print('Audit complete. Review collision examples and train/test coverage before changing preprocessing.py.')
print('Suggested decision rule: only standardise a field when variants are demonstrably equivalent and doing so improves train/test coverage or HDB match rate.')


Audit complete. Review collision examples and train/test coverage before changing preprocessing.py.
Suggested decision rule: only standardise a field when variants are demonstrably equivalent and doing so improves train/test coverage or HDB match rate.
